# Подготовка технического пайплайна DeepCluster

Этот ноутбук документирует ранний этап реализации: загрузку дообученного RuBERT-tiny2 из checkpoint, проверку CLS-эмбеддингов и подготовку загрузчика текстов для масштабного эксперимента.

Здесь используется проектная идея выборки до 500 000 товаров. Она нужна для проверки эффективности чтения parquet и кэширования row groups; итоговое контролируемое обучение и KNN-оценка выполнены отдельно в ноутбуке 8 на 25 000 товарах.

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import BertTokenizer

from pathlib import Path

LOCAL_DATA_DIR = Path("/Users/denis/Downloads")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = str(LOCAL_DATA_DIR / "dataset-electronics-5M.parquet")
CHECKPOINT_PATH = "/Users/denis/Desktop/coursework/Course_work_community_detection/models/epoch_64_encoder.pth"
TOKENIZER_DIR = "/Users/denis/Desktop/coursework/Course_work_community_detection/models/rubert-tiny2-tokenizer"
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
RANDOM_STATE = 42

print("Device:", DEVICE)
print("Dataset:", DATA_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print("Tokenizer:", TOKENIZER_DIR)

/private/tmp/deepcluster-run-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
Dataset: /Users/denis/Downloads/dataset-electronics-5M.parquet
Checkpoint: /Users/denis/Desktop/coursework/Course_work_community_detection/models/epoch_64_encoder.pth
Tokenizer: /Users/denis/Desktop/coursework/Course_work_community_detection/models/rubert-tiny2-tokenizer


In [2]:
tokenizer = BertTokenizer.from_pretrained(
    TOKENIZER_DIR,
    local_files_only=True
)

print("Tokenizer size:", len(tokenizer))

text = "Смартфон Apple iPhone 15 128GB"

print("Tokens:")
print(tokenizer.tokenize(text))

Tokenizer size: 83828
Tokens:
['Смарт', '##фон', 'Apple', 'iPhone', '15', '128', '##GB']


In [3]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint loaded")
print("Keys:", checkpoint.keys())
print("Meta:", checkpoint["meta"])

Checkpoint loaded
Keys: dict_keys(['meta', 'state_dict', 'optimizer', 'scheduler'])
Meta: {'epoch': 64, 'iter': 6400, 'time': 'Mon Sep  8 11:22:19 2025'}


In [4]:
print("Encoder будет создан и проверен в следующем блоке.")

Encoder будет создан и проверен в следующем блоке.


## Восстановление encoder из checkpoint

Сначала архитектура encoder воспроизводится в PyTorch, затем в неё загружаются веса checkpoint. Проверка одной строки подтверждает размер CLS-вектора 312 и готовность модели к извлечению признаков.

In [5]:
class BertLayer(nn.Module):
    def __init__(
        self,
        hidden_size=312,
        num_heads=12,
        intermediate_size=600
    ):
        super().__init__()

        self.attention = BertAttention(
            hidden_size,
            num_heads
        )

        self.intermediate = nn.Module()
        self.intermediate.dense = nn.Linear(
            hidden_size,
            intermediate_size
        )

        self.output = BertOutput(
            intermediate_size,
            hidden_size
        )

    def forward(self, x, attention_mask):
        x = self.attention(x, attention_mask)

        intermediate = F.gelu(
            self.intermediate.dense(x)
        )

        x = self.output(
            intermediate,
            x
        )

        return x

class BertAttention(nn.Module):
    def __init__(
        self,
        hidden_size,
        num_heads
    ):
        super().__init__()

        self.self = BertSelfAttention(
            hidden_size,
            num_heads
        )

        self.output = BertSelfOutput(
            hidden_size
        )

    def forward(self, x, attention_mask):
        attention_output = self.self(
            x,
            attention_mask
        )

        return self.output(
            attention_output,
            x
        )


class BertSelfAttention(nn.Module):
    def __init__(
        self,
        hidden_size,
        num_heads
    ):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.query = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.key = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.value = nn.Linear(
            hidden_size,
            hidden_size
        )

    def forward(self, x, attention_mask):
        batch_size, seq_len, hidden_size = x.shape

        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        q = q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        scores = torch.matmul(
            q,
            k.transpose(-1, -2)
        )

        scores = scores / np.sqrt(self.head_dim)

        scores = scores + attention_mask

        attention_probs = torch.softmax(
            scores,
            dim=-1
        )

        context = torch.matmul(
            attention_probs,
            v
        )

        context = context.transpose(
            1,
            2
        ).contiguous()

        context = context.view(
            batch_size,
            seq_len,
            hidden_size
        )

        return context


class BertSelfOutput(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()

        self.dense = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.LayerNorm = nn.LayerNorm(
            hidden_size,
            eps=1e-12
        )

    def forward(self, x, residual):
        x = self.dense(x)

        return self.LayerNorm(
            x + residual
        )


class BertOutput(nn.Module):
    def __init__(
        self,
        intermediate_size,
        hidden_size
    ):
        super().__init__()

        self.dense = nn.Linear(
            intermediate_size,
            hidden_size
        )

        self.LayerNorm = nn.LayerNorm(
            hidden_size,
            eps=1e-12
        )

    def forward(self, x, residual):
        x = self.dense(x)

        return self.LayerNorm(
            x + residual
        )


class BertEncoder(nn.Module):
    def __init__(
        self,
        vocab_size=83828,
        hidden_size=312,
        num_layers=3,
        num_heads=12,
        intermediate_size=600,
        max_position_embeddings=2048
    ):
        super().__init__()

        self.embeddings = nn.Module()

        self.embeddings.word_embeddings = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.embeddings.position_embeddings = nn.Embedding(
            max_position_embeddings,
            hidden_size
        )

        self.embeddings.token_type_embeddings = nn.Embedding(
            2,
            hidden_size
        )

        self.embeddings.LayerNorm = nn.LayerNorm(
            hidden_size,
            eps=1e-12
        )

        self.encoder = nn.Module()

        self.encoder.layer = nn.ModuleList([
            BertLayer(
                hidden_size,
                num_heads,
                intermediate_size
            )
            for _ in range(num_layers)
        ])

    def forward(
        self,
        input_ids,
        attention_mask
    ):
        batch_size, seq_len = input_ids.shape

        position_ids = torch.arange(
            seq_len,
            device=input_ids.device
        ).unsqueeze(0).expand(
            batch_size,
            -1
        )

        token_type_ids = torch.zeros_like(
            input_ids
        )

        x = (
            self.embeddings.word_embeddings(input_ids)
            + self.embeddings.position_embeddings(position_ids)
            + self.embeddings.token_type_embeddings(token_type_ids)
        )

        x = self.embeddings.LayerNorm(x)

        extended_mask = attention_mask[
            :, None, None, :
        ].float()

        extended_mask = (
            1.0 - extended_mask
        ) * -10000.0

        for layer in self.encoder.layer:
            x = layer(
                x,
                extended_mask
            )

        return x

In [6]:
encoder = BertEncoder()

state_dict = checkpoint["state_dict"]

missing, unexpected = encoder.load_state_dict(
    state_dict,
    strict=False
)

print("Missing:", missing)
print("Unexpected:", unexpected)

print(
    "Parameters:",
    sum(p.numel() for p in encoder.parameters())
)

encoder = encoder.to(DEVICE)
encoder.eval()

Missing: []
Unexpected: []
Parameters: 29096112


BertEncoder(
  (embeddings): Module(
    (word_embeddings): Embedding(83828, 312)
    (position_embeddings): Embedding(2048, 312)
    (token_type_embeddings): Embedding(2, 312)
    (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True, bias=True)
  )
  (encoder): Module(
    (layer): ModuleList(
      (0-2): 3 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=312, out_features=312, bias=True)
            (key): Linear(in_features=312, out_features=312, bias=True)
            (value): Linear(in_features=312, out_features=312, bias=True)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=312, out_features=312, bias=True)
            (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True, bias=True)
          )
        )
        (intermediate): Module(
          (dense): Linear(in_features=312, out_features=600, bias=True)
        )
        (output): Bert

In [7]:
encoder = encoder.to(DEVICE)
encoder.eval()

text = "Смартфон Apple iPhone 15 128GB"

encoded = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

input_ids = encoded["input_ids"].to(DEVICE)
attention_mask = encoded["attention_mask"].to(DEVICE)

with torch.no_grad():
    output = encoder(
        input_ids,
        attention_mask
    )

print("Input:", input_ids.shape)
print("Output:", output.shape)

embedding = output[:, 0, :]

print("Embedding:", embedding.shape)
print("Norm:", torch.norm(embedding).item())

Input: torch.Size([1, 9])
Output: torch.Size([1, 9, 312])
Embedding: torch.Size([1, 312])
Norm: 16.20451545715332


In [8]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(DATA_PATH)

print("Количество строк:", parquet_file.metadata.num_rows)

Количество строк: 4793821


## Сопоставление текста и готовых эмбеддингов

Для нескольких товаров извлекаются новые CLS-эмбеддинги encoder и сравниваются с векторами, хранящимися в parquet. Это диагностический шаг: различия возможны из-за токенизации, максимальной длины текста, pooling или состояния модели на момент предварительного расчёта эмбеддингов.

In [9]:
texts_test = []

for batch in parquet_file.iter_batches(
    batch_size=100,
    columns=["model_text"]
):
    batch_texts = batch.column("model_text").to_pylist()

    for text in batch_texts:
        if isinstance(text, bytes):
            text = text.decode("utf-8", errors="replace")

        texts_test.append(text)

        if len(texts_test) >= 5:
            break

    if len(texts_test) >= 5:
        break

for i, text in enumerate(texts_test):
    print(f"{i}: {text[:200]}")

0: Дисплей для OPPO Reno 13 F 4G In-Cell ЧерныйНе определенДисплей для OPPO Reno 13 F 4G In-Cell Черный идеально подойдет для замены Вашего разбитого дисплея. Однако, обратите внимание на важные моменты:
1: Дисплей для Tecno Spark 30C 4G ЧерныйНе определенДисплей для Tecno Spark 30C 4G Черный идеально подойдет для замены Вашего разбитого дисплея. Однако, обратите внимание на важные моменты: <br /> 1. Не 
2: Дисплей для OPPO Reno 12 F 4G/Realme 12 4G In-Cell (Premium Quality)Не определенДисплей для OPPO Reno 12 F 4G/Realme 12 4G In-Cell (Premium Quality) идеально подойдет для замены Вашего разбитого диспл
3: Дисплей для Xiaomi 13 (2211133C) OLEDНе определенДисплей для Xiaomi 13 (2211133C) OLED идеально подойдет для замены Вашего разбитого дисплея. Однако, обратите внимание на важные моменты: <br /> 1. Не 
4: Дисплей для Samsung Galaxy S23 в рамке Черный In-CellНе определенДисплей для Samsung Galaxy S23 в рамке Черный In-Cell идеально подойдет для замены Вашего разбитого дисплея. Однако,

In [10]:
def get_embeddings(
    texts,
    batch_size=8,
    max_length=512
):
    encoder.eval()

    embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(DEVICE)
        attention_mask = encoded["attention_mask"].to(DEVICE)

        with torch.no_grad():
            output = encoder(
                input_ids,
                attention_mask
            )

        batch_embeddings = output[:, 0, :]

        embeddings.append(
            batch_embeddings.cpu().numpy()
        )

    return np.concatenate(embeddings, axis=0)

In [11]:
test_embeddings = get_embeddings(
    texts_test,
    batch_size=5,
    max_length=512
)

print("Shape:", test_embeddings.shape)
print("First embedding:", test_embeddings[0][:10])

Shape: (5, 312)
First embedding: [-1.2849953  -0.04631741 -1.1420535   0.32879013  0.24574369 -2.050823
 -0.22076017 -0.31604886  0.02441853 -0.5452219 ]


In [12]:
embeddings_test_raw = []

for batch in parquet_file.iter_batches(
    batch_size=5,
    columns=["embedding"]
):
    embeddings_test_raw.extend(
        batch.column("embedding").to_pylist()
    )
    break

print("Количество:", len(embeddings_test_raw))
print("Размер первого:", len(embeddings_test_raw[0]))

Количество: 5
Размер первого: 3122


In [13]:
def decode_embedding(data):
    data = bytes(data)

    if len(data) != 3122:
        raise ValueError(
            f"Неожиданный размер embedding: {len(data)}"
        )

    values = np.empty(
        312,
        dtype=np.float64
    )

    for i in range(312):
        start = 2 + i * 10

        values[i] = np.frombuffer(
            data[start:start + 8],
            dtype="<f8"
        )[0]

    return values.astype(np.float32)


original_embeddings = np.stack([
    decode_embedding(x)
    for x in embeddings_test_raw
])

print(
    "Shape:",
    original_embeddings.shape
)

print(
    "First embedding:",
    original_embeddings[0][:10]
)

Shape: (5, 312)
First embedding: [-0.08203125 -0.00546265 -0.07910156  0.02307129  0.00726318 -0.15332031
 -0.02185059 -0.00970459  0.00848389 -0.04150391]


In [14]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = np.diag(
    cosine_similarity(
        test_embeddings,
        original_embeddings
    )
)

for i, similarity in enumerate(similarities):
    print(
        f"Text {i}: cosine similarity = {similarity:.6f}"
    )

Text 0: cosine similarity = 0.995285
Text 1: cosine similarity = 0.996131
Text 2: cosine similarity = 0.995412
Text 3: cosine similarity = 0.996360
Text 4: cosine similarity = 0.996685


Переходим к DeepCluster

## План масштабной выборки

Выборка из 500 000 товаров пропорциональна распределению 122 категорий. Этот объём нужен не для финальной оценки качества, а для проверки того, что реализация способна обслуживать большой каталог без загрузки всего parquet в память.

In [15]:
SAMPLE_SIZE = 25_000
N_CLUSTERS = 10
MAX_LENGTH = 128
BATCH_SIZE = 32
LEARNING_RATE = 1e-5

print("Sample size:", SAMPLE_SIZE)
print("Clusters:", N_CLUSTERS)
print("Max length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)

Sample size: 25000
Clusters: 10
Max length: 128
Batch size: 32


In [16]:
pf = pq.ParquetFile(DATA_PATH)

category_counts = {}

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["category_id"]
):
    categories = batch.column("category_id").to_numpy()

    unique, counts = np.unique(
        categories,
        return_counts=True
    )

    for category, count in zip(unique, counts):
        category = int(category)
        category_counts[category] = (
            category_counts.get(category, 0) + int(count)
        )

print("Categories:", len(category_counts))
print("Total rows:", sum(category_counts.values()))

Categories: 122
Total rows: 4793821


In [17]:
rng = np.random.default_rng(RANDOM_STATE)

indices_by_category = {
    category: []
    for category in category_counts
}

row_start = 0

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["category_id"]
):
    categories = batch.column("category_id").to_numpy()

    for local_idx, category in enumerate(categories):
        category = int(category)

        indices_by_category[category].append(
            row_start + local_idx
        )

    row_start += len(categories)

sample_indices = []

for category, indices in indices_by_category.items():
    sample_n = round(
        SAMPLE_SIZE *
        len(indices) /
        row_start
    )

    sample_n = min(
        sample_n,
        len(indices)
    )

    selected = rng.choice(
        indices,
        size=sample_n,
        replace=False
    )

    sample_indices.extend(selected)

sample_indices = np.array(
    sample_indices,
    dtype=np.int64
)

rng.shuffle(sample_indices)

if len(sample_indices) > SAMPLE_SIZE:
    sample_indices = rng.choice(
        sample_indices,
        size=SAMPLE_SIZE,
        replace=False
    )

print("Sample:", len(sample_indices))

Sample: 25000


## Доступ к текстам по индексам parquet

В следующих блоках строится индекс row groups. Он позволяет быстро найти текст товара по глобальному номеру строки и кэшировать уже прочитанные группы. Это устраняет повторное чтение одного и того же фрагмента parquet при формировании batch.

In [18]:
from torch.utils.data import Dataset, DataLoader


class DeepClusterDataset(Dataset):
    def __init__(
        self,
        parquet_path,
        indices,
        pseudo_labels,
        tokenizer,
        max_length=512
    ):
        self.parquet_path = parquet_path
        self.indices = np.asarray(
            indices,
            dtype=np.int64
        )
        self.pseudo_labels = np.asarray(
            pseudo_labels,
            dtype=np.int64
        )
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.pf = pq.ParquetFile(
            parquet_path
        )

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        row_idx = self.indices[idx]

        table = self.pf.read_row_group(
            self._find_row_group(row_idx),
            columns=["model_text"]
        )

        text = table["model_text"][
            row_idx - self._row_group_start
        ].as_py()

        if isinstance(text, bytes):
            text = text.decode(
                "utf-8",
                errors="replace"
            )

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": torch.tensor(
                self.pseudo_labels[idx],
                dtype=torch.long
            )
        }

Создаём индекс row groups

In [19]:
pf = pq.ParquetFile(DATA_PATH)

row_group_starts = []
row_group_sizes = []

current_start = 0

for i in range(pf.num_row_groups):
    size = pf.metadata.row_group(i).num_rows

    row_group_starts.append(current_start)
    row_group_sizes.append(size)

    current_start += size

row_group_starts = np.array(row_group_starts)
row_group_sizes = np.array(row_group_sizes)

print("Row groups:", pf.num_row_groups)
print("Total rows:", current_start)

Row groups: 2108
Total rows: 4793821


In [20]:
def get_row_group(row_idx):
    group = np.searchsorted(
        row_group_starts,
        row_idx,
        side="right"
    ) - 1

    return group, row_idx - row_group_starts[group]

In [21]:
for idx in [0, 100, 100000, 1_000_000, 4_000_000]:
    group, local_idx = get_row_group(idx)
    print(
        idx,
        "→ row_group:",
        group,
        "local:",
        local_idx
    )

0 → row_group: 0 local: 0
100 → row_group: 0 local: 100
100000 → row_group: 38 local: 919
1000000 → row_group: 438 local: 487
4000000 → row_group: 1747 local: 243


In [22]:
class DeepClusterDataset(Dataset):
    def __init__(
        self,
        parquet_path,
        indices,
        pseudo_labels,
        tokenizer,
        row_group_starts,
        max_length=512
    ):
        self.indices = np.asarray(indices)
        self.pseudo_labels = np.asarray(pseudo_labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.pf = pq.ParquetFile(parquet_path)
        self.row_group_starts = row_group_starts

        self.row_groups = np.searchsorted(
            row_group_starts,
            self.indices,
            side="right"
        ) - 1

        self.cache = {}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        row_idx = self.indices[idx]
        group = self.row_groups[idx]
        local_idx = row_idx - self.row_group_starts[group]

        if group not in self.cache:
            table = self.pf.read_row_group(
                int(group),
                columns=["model_text"]
            )
            self.cache[group] = table["model_text"]

        text = self.cache[group][local_idx].as_py()

        if isinstance(text, bytes):
            text = text.decode(
                "utf-8",
                errors="replace"
            )

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": torch.tensor(
                self.pseudo_labels[idx],
                dtype=torch.long
            )
        }

In [23]:
test_labels = np.zeros(
    len(sample_indices),
    dtype=np.int64
)

dataset_test = DeepClusterDataset(
    DATA_PATH,
    sample_indices,
    test_labels,
    tokenizer,
    row_group_starts,
    max_length=MAX_LENGTH
)

In [24]:
item = dataset_test[0]

print("input_ids:", item["input_ids"].shape)
print("attention_mask:", item["attention_mask"].shape)
print("label:", item["label"])

input_ids: torch.Size([128])
attention_mask: torch.Size([128])
label: tensor(0)


In [25]:
for idx in [0, 100, 1000, 10000, min(24_999, len(dataset_test) - 1)]:
    item = dataset_test[idx]

    print(
        idx,
        item["input_ids"].shape,
        item["attention_mask"].sum().item()
    )

0 torch.Size([128]) 128
100 torch.Size([128]) 128
1000 torch.Size([128]) 128
10000 torch.Size([128]) 128
24999 torch.Size([128]) 128


Функция генерации embeddings

In [26]:
def generate_embeddings(
    texts,
    batch_size=16,
    max_length=512
):
    encoder.eval()

    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(DEVICE)
        attention_mask = encoded["attention_mask"].to(DEVICE)

        with torch.no_grad():
            output = encoder(
                input_ids,
                attention_mask
            )

        embeddings = output[:, 0, :]
        all_embeddings.append(
            embeddings.cpu().numpy()
        )

        if start % 10000 == 0:
            print(
                f"{start:,}/{len(texts):,}"
            )

    return np.concatenate(
        all_embeddings,
        axis=0
    )

In [27]:
class EmbeddingDataset(Dataset):
    def __init__(
        self,
        parquet_path,
        indices,
        tokenizer,
        row_group_starts,
        max_length=512
    ):
        self.indices = np.asarray(indices)
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.pf = pq.ParquetFile(parquet_path)
        self.row_group_starts = row_group_starts

        self.row_groups = np.searchsorted(
            row_group_starts,
            self.indices,
            side="right"
        ) - 1

        self.cache = {}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        row_idx = self.indices[idx]
        group = self.row_groups[idx]

        local_idx = (
            row_idx -
            self.row_group_starts[group]
        )

        if group not in self.cache:
            table = self.pf.read_row_group(
                int(group),
                columns=["model_text"]
            )
            self.cache[group] = table["model_text"]

        text = self.cache[group][local_idx].as_py()

        if isinstance(text, bytes):
            text = text.decode(
                "utf-8",
                errors="replace"
            )

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0)
        }

In [28]:
embedding_dataset = EmbeddingDataset(
    DATA_PATH,
    sample_indices,
    tokenizer,
    row_group_starts,
    max_length=MAX_LENGTH
)

embedding_loader = DataLoader(
    embedding_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Dataset size:", len(embedding_dataset))

Dataset size: 25000


In [29]:
MAX_LENGTH = 128
BATCH_SIZE = 32

print("Max length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)

Max length: 128
Batch size: 32


In [30]:
embedding_dataset = EmbeddingDataset(
    DATA_PATH,
    sample_indices,
    tokenizer,
    row_group_starts,
    max_length=MAX_LENGTH
)

embedding_loader = DataLoader(
    embedding_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Dataset size:", len(embedding_dataset))
print("Batches:", len(embedding_loader))

Dataset size: 25000
Batches: 782


In [31]:
BATCH_SIZE = 32

TEST_SIZE = 100

In [32]:
TEST_SIZE = 100
BATCH_SIZE = 32

embedding_dataset_test = torch.utils.data.Subset(
    embedding_dataset,
    range(TEST_SIZE)
)

embedding_loader = DataLoader(
    embedding_dataset_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Dataset size:", len(embedding_loader.dataset))
print("Batches:", len(embedding_loader))

Dataset size: 100
Batches: 4


In [33]:
encoder.eval()

embeddings = []

total_batches = len(embedding_loader)
total_samples = len(embedding_loader.dataset)

for batch_idx, batch in enumerate(embedding_loader, start=1):

    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)

    with torch.no_grad():
        output = encoder(
            input_ids,
            attention_mask
        )

    embeddings.append(
        output[:, 0, :].cpu().numpy()
    )

    processed = min(
        batch_idx * BATCH_SIZE,
        total_samples
    )

    print(
        f"\rБатч: {batch_idx}/{total_batches} | "
        f"Обработано: {processed}/{total_samples} | "
        f"Прогресс: {processed / total_samples * 100:.1f}%",
        end="",
        flush=True
    )

X_deep = np.concatenate(embeddings, axis=0)

print()
print("Shape:", X_deep.shape)

Батч: 1/4 | Обработано: 32/100 | Прогресс: 32.0%

Батч: 2/4 | Обработано: 64/100 | Прогресс: 64.0%

Батч: 3/4 | Обработано: 96/100 | Прогресс: 96.0%

Батч: 4/4 | Обработано: 100/100 | Прогресс: 100.0%


Shape: (100, 312)


In [34]:
original_embeddings_raw = []

for batch in parquet_file.iter_batches(
    batch_size=100,
    columns=["embedding"]
):
    original_embeddings_raw.extend(
        batch.column("embedding").to_pylist()
    )

    if len(original_embeddings_raw) >= 100:
        break

original_embeddings = np.array([
    decode_embedding(x)
    for x in original_embeddings_raw[:100]
])

print("Original:", original_embeddings.shape)
print("New:", X_deep.shape)

Original: (100, 312)
New: (100, 312)


In [35]:
similarities = []

for i in range(len(X_deep)):
    sim = cosine_similarity(
        X_deep[i:i+1],
        original_embeddings[i:i+1]
    )[0, 0]

    similarities.append(sim)

print("Средняя cosine similarity:", np.mean(similarities))
print("Минимальная:", np.min(similarities))
print("Максимальная:", np.max(similarities))

Средняя cosine similarity: 0.07295997
Минимальная: -0.1881079
Максимальная: 0.45045236


In [36]:
text = texts_test[0]

encoded = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

input_ids = encoded["input_ids"].to(DEVICE)
attention_mask = encoded["attention_mask"].to(DEVICE)

with torch.no_grad():
    output = encoder(
        input_ids,
        attention_mask
    )

new_embedding = output[:, 0, :].cpu().numpy()[0]

original_embedding = decode_embedding(
    embeddings_test_raw[0]
)

similarity = cosine_similarity(
    new_embedding.reshape(1, -1),
    original_embedding.reshape(1, -1)
)[0, 0]

print("Text:", text[:200])
print("New embedding:", new_embedding.shape)
print("Original embedding:", original_embedding.shape)
print("Cosine similarity:", similarity)

Text: Дисплей для OPPO Reno 13 F 4G In-Cell ЧерныйНе определенДисплей для OPPO Reno 13 F 4G In-Cell Черный идеально подойдет для замены Вашего разбитого дисплея. Однако, обратите внимание на важные моменты:
New embedding: (312,)
Original embedding: (312,)
Cosine similarity: 0.9922245


In [37]:
texts_100 = []
original_embeddings_raw = []

for batch in parquet_file.iter_batches(
    batch_size=100,
    columns=["model_text", "embedding"]
):
    texts_100 = batch.column("model_text").to_pylist()
    original_embeddings_raw = batch.column("embedding").to_pylist()
    break

texts_100 = [
    x.decode("utf-8", errors="replace")
    if isinstance(x, bytes)
    else x
    for x in texts_100
]

original_embeddings = np.array([
    decode_embedding(x)
    for x in original_embeddings_raw
])

print("Texts:", len(texts_100))
print("Original embeddings:", original_embeddings.shape)

Texts: 100
Original embeddings: (100, 312)


In [38]:
from torch.utils.data import Dataset, DataLoader


class TextDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.texts = texts
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=128,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0)
        }


test_dataset = TextDataset(
    texts_100,
    tokenizer
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Dataset:", len(test_dataset))
print("Batches:", len(test_loader))

Dataset: 100
Batches: 7


In [39]:
encoder.eval()

new_embeddings = []

total_batches = len(test_loader)
total_samples = len(test_dataset)

for batch_idx, batch in enumerate(test_loader, start=1):

    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)

    with torch.no_grad():
        output = encoder(
            input_ids,
            attention_mask
        )

    new_embeddings.append(
        output[:, 0, :].cpu().numpy()
    )

    processed = min(
        batch_idx * 16,
        total_samples
    )

    print(
        f"\rБатч: {batch_idx}/{total_batches} | "
        f"Обработано: {processed}/{total_samples} | "
        f"Прогресс: {processed / total_samples * 100:.1f}%",
        end="",
        flush=True
    )

X_deep = np.concatenate(
    new_embeddings,
    axis=0
)

print()
print("New embeddings:", X_deep.shape)

Батч: 1/7 | Обработано: 16/100 | Прогресс: 16.0%

Батч: 2/7 | Обработано: 32/100 | Прогресс: 32.0%

Батч: 3/7 | Обработано: 48/100 | Прогресс: 48.0%

Батч: 4/7 | Обработано: 64/100 | Прогресс: 64.0%

Батч: 5/7 | Обработано: 80/100 | Прогресс: 80.0%

Батч: 6/7 | Обработано: 96/100 | Прогресс: 96.0%

Батч: 7/7 | Обработано: 100/100 | Прогресс: 100.0%


New embeddings: (100, 312)


In [40]:
similarities = np.sum(
    X_deep * original_embeddings,
    axis=1
) / (
    np.linalg.norm(X_deep, axis=1)
    * np.linalg.norm(original_embeddings, axis=1)
)

print("Средняя cosine similarity:", similarities.mean())
print("Минимальная:", similarities.min())
print("Максимальная:", similarities.max())

Средняя cosine similarity: 0.9882128
Минимальная: 0.9279723
Максимальная: 0.99792325


In [41]:
SAMPLE_SIZE = 25_000

sample_indices = np.asarray(sample_indices)

texts = []

for batch in parquet_file.iter_batches(
    columns=["model_text"],
    batch_size=10_000
):
    batch_texts = batch.column("model_text").to_pylist()
    texts.extend(batch_texts)

    if len(texts) >= SAMPLE_SIZE:
        break

texts = np.array([
    x.decode("utf-8", errors="replace")
    if isinstance(x, bytes)
    else x
    for x in texts[:SAMPLE_SIZE]
], dtype=object)

print("Texts:", texts.shape)

Texts: (25000,)


In [42]:
BATCH_SIZE = 32

deep_dataset = TextDataset(
    texts,
    tokenizer
)

deep_loader = DataLoader(
    deep_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Dataset:", len(deep_dataset))
print("Batches:", len(deep_loader))

Dataset: 25000
Batches: 782


In [43]:
encoder.eval()

deep_embeddings = []

total_batches = len(deep_loader)
total_samples = len(deep_dataset)

for batch_idx, batch in enumerate(deep_loader, start=1):

    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)

    with torch.no_grad():
        output = encoder(
            input_ids,
            attention_mask
        )

    deep_embeddings.append(
        output[:, 0, :].cpu().numpy()
    )

    processed = min(
        batch_idx * BATCH_SIZE,
        total_samples
    )

    print(
        f"\rБатч: {batch_idx}/{total_batches} | "
        f"Обработано: {processed}/{total_samples} | "
        f"Прогресс: {processed / total_samples * 100:.1f}%",
        end="",
        flush=True
    )

X_deep = np.concatenate(
    deep_embeddings,
    axis=0
)

print()
print("Shape:", X_deep.shape)

Батч: 1/782 | Обработано: 32/25000 | Прогресс: 0.1%

Батч: 2/782 | Обработано: 64/25000 | Прогресс: 0.3%

Батч: 3/782 | Обработано: 96/25000 | Прогресс: 0.4%

Батч: 4/782 | Обработано: 128/25000 | Прогресс: 0.5%

Батч: 5/782 | Обработано: 160/25000 | Прогресс: 0.6%

...

Батч: 779/782 | Обработано: 24928/25000 | Прогресс: 99.7%

Батч: 780/782 | Обработано: 24960/25000 | Прогресс: 99.8%

Батч: 781/782 | Обработано: 24992/25000 | Прогресс: 100.0%

Батч: 782/782 | Обработано: 25000/25000 | Прогресс: 100.0%


Shape: (25000, 312)


In [44]:
print("Shape:", X_deep.shape)
print("NaN:", np.isnan(X_deep).sum())
print("Inf:", np.isinf(X_deep).sum())

norms = np.linalg.norm(X_deep, axis=1)

print("Norm min:", norms.min())
print("Norm mean:", norms.mean())
print("Norm max:", norms.max())

Shape: (25000, 312)
NaN: 0


Inf: 0
Norm min: 11.945552
Norm mean: 15.072707
Norm max: 16.511423


In [45]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

In [46]:
pca_deep = PCA(
    n_components=256,
    whiten=True,
    random_state=RANDOM_STATE
)

X_deep_pca = pca_deep.fit_transform(X_deep)

X_deep_cluster = normalize(
    X_deep_pca,
    norm="l2"
)

print("Before:", X_deep.shape)
print("After PCA:", X_deep_pca.shape)
print("After L2:", X_deep_cluster.shape)

Before: (25000, 312)
After PCA: (25000, 256)
After L2: (25000, 256)


In [47]:
from sklearn.cluster import KMeans

N_CLUSTERS = 10

kmeans_deep = KMeans(
    n_clusters=N_CLUSTERS,
    n_init=10,
    max_iter=300,
    random_state=RANDOM_STATE
)

labels_deep = kmeans_deep.fit_predict(
    X_deep_cluster
)

print("Labels:", labels_deep.shape)
print("Clusters:", np.unique(labels_deep))

Labels: (25000,)
Clusters: [0 1 2 3 4 5 6 7 8 9]


In [48]:
table = parquet_file.read(
    columns=["category_id"]
)

selected = table.take(
    sample_indices
)

y = np.array(
    selected.column("category_id").to_pylist()
)

print("y shape:", y.shape)
print("Unique categories:", len(np.unique(y)))

y shape: (25000,)
Unique categories: 120


In [49]:
y_deep = y.copy()

print("Текстов в выборке:", len(y_deep))
print("y_deep:", y_deep.shape)
print("Уникальных категорий:", len(np.unique(y_deep)))

Текстов в выборке: 25000
y_deep: (25000,)
Уникальных категорий: 120


In [50]:
print("X_deep:", X_deep.shape)
print("labels_deep:", labels_deep.shape)
print("sample_indices:", sample_indices.shape)
print("sample_indices min:", sample_indices.min())
print("sample_indices max:", sample_indices.max())

X_deep: (25000, 312)
labels_deep: (25000,)
sample_indices: (25000,)
sample_indices min: 7
sample_indices max: 4793574


In [51]:
print("texts:", texts.shape)
print("First text:", texts[0])
print("Last text:", texts[-1])

texts: (25000,)
First text: Дисплей для OPPO Reno 13 F 4G In-Cell ЧерныйНе определенДисплей для OPPO Reno 13 F 4G In-Cell Черный идеально подойдет для замены Вашего разбитого дисплея. Однако, обратите внимание на важные моменты: <br /> 1. Не забудьте выключить телефон перед заменой дисплейного модуля.<br /> 2. Проведите предварительную проверку дисплея, просто подключив его к телефону, убедитесь, что всё корректно работает, и только после этого приступайте к установке. Этот пункт крайне важен так как: гарантия НЕ распространяется на дисплеи после установки.<br /> 3. Если на задней стороне дисплея присутствует двухсторонний скотч для фиксации шлейфа, обязательно приклейте шлейф с его помощью, в противном случае тачскрин будет работать некорректно.<br /> 4. Не прилагайте усилий при установке, при правильной установке, дисплей помещается в посадочное место без усилий.<br /> 5. При прокладке шлейфов ни в коем случае не допускайте заломов.<br /> 6. Если под старым дисплеем были какие-либо п

## Связь с итоговым экспериментом

Этот ноутбук решает инженерную задачу: как надёжно восстановить encoder и прочитать большой стратифицированный набор текстов. Полный цикл DeepCluster с переобучением BERT, пересчётом кластеров, KNN-baseline, графиками loss и validation-метрик реализован в ноутбуке 8. Поэтому результаты промежуточных диагностических ячеек не интерпретируются как итоговая оценка качества эмбеддингов.